# Notebook 07 — Static Feature Table

**Purpose**: Collapse the loan-month panel to one row per `LOAN_ID` with the origination-time features needed for M1 (baseline) and M2 (climate-augmented).

**Inputs**
- `fl_loan_month_clean.parquet`
- `fl_default_labels.parquet`

**Output**
- `fl_static_features.parquet` — one row per loan, all origination-time variables + labels

**Method notes**
- Origination-time features are constant across the loan-month panel; take the earliest observation per loan.
- `ORIG_DATE` in Fannie Mae is `MMYYYY`; we extract `ORIG_YEAR` for the climate merge.
- `ZIP` in Fannie Mae is already 3-digit (privacy convention); store as zero-padded string.
- `ORIG_LTV` is retained here for descriptive statistics but will be dropped in M1 (collinear with `ORIG_CLTV`).


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("your/data/path/here")

IN_LM   = DATA_DIR / "fl_loan_month_clean.parquet"
IN_LBL  = DATA_DIR / "fl_default_labels.parquet"
OUT_STC = DATA_DIR / "fl_static_features.parquet"

assert IN_LM.exists() and IN_LBL.exists()


## 1. Load loan-month panel and labels

In [2]:
df  = pd.read_parquet(IN_LM)
lbl = pd.read_parquet(IN_LBL)
print(f"Loan-month rows: {len(df):,}")
print(f"Unique loans   : {df['LOAN_ID'].nunique():,}")
print(f"Labels rows    : {len(lbl):,}")


Loan-month rows: 1,222,563
Unique loans   : 19,625
Labels rows    : 19,625


## 2. Collapse to one row per LOAN_ID

In [3]:
static_cols = [
    'ORIG_DATE',       # MMYYYY
    'ORIG_RATE',
    'ORIG_UPB',
    'ORIG_TERM',
    'ORIG_LTV',
    'ORIG_CLTV',
    'NUM_BORR',
    'DTI',
    'CSCORE_B',
    'FIRST_FLAG',
    'PURPOSE',
    'PROP',
    'NO_UNITS',
    'OCC_STAT',
    'STATE',
    'ZIP',             # 3-digit
    'MI_PCT',
    'CHANNEL',
]

static = (df.sort_values(['LOAN_ID', 'PERIOD_YYYYMM'])
            .groupby('LOAN_ID')[static_cols]
            .first()
            .reset_index())

print(f"Static rows: {len(static):,}")
static.head()


Static rows: 19,625


,LOAN_ID,ORIG_DATE,ORIG_RATE,ORIG_UPB,ORIG_TERM,ORIG_LTV,ORIG_CLTV,NUM_BORR,DTI,CSCORE_B,FIRST_FLAG,PURPOSE,PROP,NO_UNITS,OCC_STAT,STATE,ZIP,MI_PCT,CHANNEL
0,100014590941,112016,4.750,90000.0,360,75,75,2,41.0,786.0,N,P,CO,1,I,FL,331,NaN,C
1,100039728035,12017,4.875,152000.0,360,95,105,2,38.0,651.0,Y,P,PU,1,P,FL,336,16.0,C
2,100177392531,112016,4.875,140000.0,360,85,85,1,3.0,818.0,N,P,SF,1,I,FL,339,12.0,R
3,100244308468,12017,4.750,107000.0,360,92,105,1,39.0,707.0,Y,P,SF,1,P,FL,338,16.0,C
4,100281770072,122016,4.250,75000.0,360,49,49,1,49.0,696.0,N,P,CO,1,P,FL,330,NaN,B


## 3. Derive fields for downstream use

In [4]:
# ORIG_YEAR from MMYYYY
def orig_year(x):
    s = str(int(x)).zfill(6)
    return int(s[2:])

static['ORIG_YEAR'] = static['ORIG_DATE'].apply(orig_year)

# FIRST_FLAG as 0/1
static['FIRST_FLAG'] = (static['FIRST_FLAG'].astype(str).str.upper().str.strip() == 'Y').astype(int)

# MI_PCT: missing means no MI, so 0
static['MI_PCT'] = pd.to_numeric(static['MI_PCT'], errors='coerce').fillna(0)

# ZIP3 as zero-padded string for the crosswalk merge
static['ZIP3'] = static['ZIP'].astype(str).str.replace(r'\D', '', regex=True).str.zfill(3).str[:3]

print(static[['ORIG_YEAR', 'ZIP3', 'FIRST_FLAG', 'MI_PCT']].head())
print(f"\nOrigination year distribution:\n{static['ORIG_YEAR'].value_counts().sort_index()}")


   ORIG_YEAR ZIP3  FIRST_FLAG  MI_PCT
0       2016  331           0     0.0
1       2017  336           1    16.0
2       2016  339           0    12.0
3       2017  338           1    16.0
4       2016  330           0     0.0

Origination year distribution:
ORIG_YEAR
2015        1
2016     8797
2017    10827
Name: count, dtype: int64


## 4. Merge labels in

In [5]:
static = static.merge(lbl, on='LOAN_ID', how='left')

# Any loans without a label? (Shouldn't happen but check)
missing_lbl = static['default_180dpd'].isna().sum()
print(f"Loans missing label: {missing_lbl}")

print(f"\nOverall default rate: {static['default_180dpd'].mean():.4f}")
print(f"Overall forbearance rate: {static['covid_forbearance'].mean():.4f}")


Loans missing label: 0

Overall default rate: 0.0285
Overall forbearance rate: 0.0427


## 5. Missingness check on features

In [9]:
static['HAS_MI'] = (static['MI_PCT'] > 0).astype(int)

In [6]:
na_summary = static.isna().sum()
na_summary = na_summary[na_summary > 0].sort_values(ascending=False)
if len(na_summary) == 0:
    print("No missing values in static feature table.")
else:
    print("Missing values:")
    print(na_summary)


No missing values in static feature table.


## 6. Descriptive statistics of key numeric features

In [7]:
num_cols = ['CSCORE_B', 'DTI', 'ORIG_LTV', 'ORIG_CLTV', 'ORIG_RATE', 'ORIG_UPB', 'MI_PCT']
static[num_cols].describe().round(2)


,CSCORE_B,DTI,ORIG_LTV,ORIG_CLTV,ORIG_RATE,ORIG_UPB,MI_PCT
count,19625.00,19625.00,19625.00,19625.00,19625.00,19625.00,19625.00
mean,750.77,35.67,80.61,81.74,4.28,202934.73,10.42
std,47.19,8.64,14.50,15.25,0.48,97034.61,13.01
min,620.00,1.00,8.00,8.00,2.25,20000.00,0.00
25%,715.00,30.00,75.00,75.00,3.99,127000.00,0.00
50%,761.00,37.00,80.00,80.00,4.38,185000.00,0.00
75%,791.00,43.00,93.00,95.00,4.62,267000.00,25.00
max,832.00,50.00,97.00,105.00,6.12,540000.00,35.00


## 7. Save

In [10]:
static.to_parquet(OUT_STC, index=False)
print(f"Saved: {OUT_STC}")
print(f"Shape: {static.shape}")


Saved: E:\Financial Mathsmatics Master\Dissertation\datasets\Data Processed\fl_static_features.parquet
Shape: (19625, 24)
